In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import vmap, jvp
import numpy as np

In [2]:
# ============================================================
# Configuration
# ============================================================

DEFAULT_DTYPE = torch.float32
EPS64 = 1e-12
EPS32 = 1e-7


def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def get_eps(dtype):
    return EPS32 if dtype == torch.float32 else EPS64


def recommended_dtype_for_device(device):
    if isinstance(device, str):
        device = torch.device(device)
    if device.type in ("cuda", "mps"):
        return torch.float32
    return torch.float64


# ============================================================
# S^2 geometry utilities
# ============================================================

def normalize_torch(x, eps=None):
    if eps is None:
        eps = get_eps(x.dtype)
    nrm = torch.linalg.norm(x, dim=-1, keepdim=True)
    return x / torch.clamp(nrm, min=eps)


def tangent_project_torch(x, v):
    return v - torch.sum(x * v, dim=-1, keepdim=True) * x


def sphere_exp_map_torch(x, v, eps=None):
    if eps is None:
        eps = get_eps(x.dtype)

    x = normalize_torch(x, eps=eps)
    v = tangent_project_torch(x, v)
    r = torch.linalg.norm(v, dim=-1, keepdim=True)

    sr_over_r = torch.where(
        r > eps,
        torch.sin(r) / r,
        1.0 - (r ** 2) / 6.0
    )

    y = torch.cos(r) * x + sr_over_r * v
    return normalize_torch(y, eps=eps)


def sphere_distance_torch(x, y, euclidean=True, eps=None):
    if eps is None:
        eps = get_eps(x.dtype)

    x = normalize_torch(x, eps=eps)
    y = normalize_torch(y, eps=eps)

    dot = torch.sum(x * y, dim=-1).clamp(-1.0, 1.0)

    if euclidean:
        return torch.sqrt(torch.clamp(2.0 * (1.0 - dot), min=0.0))
    else:
        sin_theta = torch.linalg.norm(x - dot.unsqueeze(-1) * y, dim=-1)
        return torch.atan2(sin_theta, dot)


def parallel_transport_S2_torch(x, y, V, eps=None):
    if eps is None:
        eps = get_eps(x.dtype)

    x = normalize_torch(x, eps=eps)
    y = normalize_torch(y, eps=eps)

    dot = torch.sum(x * y, dim=-1, keepdim=True)
    denom = torch.clamp(1.0 + dot, min=eps)

    yTV = torch.sum(y.unsqueeze(-1) * V, dim=-2, keepdim=True)
    corr = (yTV / denom.unsqueeze(-1)) * (x + y).unsqueeze(-1)
    W = V - corr

    W = W - torch.sum(y.unsqueeze(-1) * W, dim=-2, keepdim=True) * y.unsqueeze(-1)
    return W


def orthonormalize_frames_torch(U, eps=None):
    if eps is None:
        eps = get_eps(U.dtype)

    v0 = U[..., :, 0]
    n0 = torch.linalg.norm(v0, dim=-1, keepdim=True)
    q0 = v0 / torch.clamp(n0, min=eps)

    v1 = U[..., :, 1]
    proj = torch.sum(q0 * v1, dim=-1, keepdim=True)
    v1 = v1 - proj * q0
    n1 = torch.linalg.norm(v1, dim=-1, keepdim=True)
    q1 = v1 / torch.clamp(n1, min=eps)

    return torch.stack([q0, q1], dim=-1)


def make_tangent_frame_S2_batch_torch(X, eps=None):
    if eps is None:
        eps = get_eps(X.dtype)

    X = normalize_torch(X, eps=eps)

    use_x = torch.abs(X[..., 0]) < 0.9
    A_x = torch.tensor([1.0, 0.0, 0.0], dtype=X.dtype, device=X.device)
    A_y = torch.tensor([0.0, 1.0, 0.0], dtype=X.dtype, device=X.device)
    A = torch.where(use_x.unsqueeze(-1), A_x, A_y)

    e1 = tangent_project_torch(X, A)
    e1 = normalize_torch(e1, eps=eps)

    e2 = torch.cross(X, e1, dim=-1)
    e2 = normalize_torch(e2, eps=eps)

    return torch.stack([e1, e2], dim=-1)


# ============================================================
# Default f_fn examples
# ============================================================

def default_gaussian_kernel_f(X, y, alpha=5.0, euclidean=True):
    """
    X: (B,P,3)
    y: (3,) or broadcastable to (B,P,3)
    returns: (B,P)
    """
    dist = sphere_distance_torch(X, y, euclidean=euclidean)
    return torch.exp(-alpha * dist * dist)


def indicator_cap_f(X, y, radius=0.5, euclidean=False):
    """
    Example non-smooth test function:
      f(X,y) = 1{ d(X,y) <= radius }
    """
    dist = sphere_distance_torch(X, y, euclidean=euclidean)
    return (dist <= radius).to(X.dtype)


# ============================================================
# One chunk kernel
# ============================================================

@torch.no_grad()
def _bel_chunk_kernel_torch(
    X0, y, t,
    f_fn,
    chunk_size,
    n_steps,
    generator=None,
    reorthonormalize_every=1,
    f_kwargs=None,
):
    """
    Computes chunk contributions for BEL estimator with arbitrary payoff f_fn.

    Inputs:
      X0: (B,3)
      y:  (3,) or broadcastable target object consumed by f_fn
      f_fn: callable with signature
              f_fn(X, y, **f_kwargs) -> (B,P)
            where X has shape (B,P,3)
    Returns:
      sum_f:  (B,)
      sum_fI: (B,2)
    """
    if f_kwargs is None:
        f_kwargs = {}

    dtype = X0.dtype
    device = X0.device
    eps = get_eps(dtype)

    X0 = normalize_torch(X0, eps=eps)
    y = torch.as_tensor(y, dtype=dtype, device=device)

    B = X0.shape[0]
    P = int(chunk_size)

    t = torch.as_tensor(t, dtype=dtype, device=device)
    dt = t / n_steps
    sqrt_dt = torch.sqrt(dt)

    U0 = make_tangent_frame_S2_batch_torch(X0, eps=eps)

    X = X0[:, None, :].expand(B, P, 3).clone()
    U = U0[:, None, :, :].expand(B, P, 3, 2).clone()
    I = torch.zeros(B, P, 2, dtype=dtype, device=device)

    for k in range(n_steps):
        tk = (k + 0.5) * dt
        M_scalar = torch.exp(-0.5 * tk)

        dW = torch.randn(
            B, P, 2,
            dtype=dtype,
            device=device,
            generator=generator
        ) * sqrt_dt

        V = torch.einsum("bpij,bpj->bpi", U, dW)

        X_new = sphere_exp_map_torch(X, V, eps=eps)
        U_new = parallel_transport_S2_torch(X, X_new, U, eps=eps)

        if reorthonormalize_every > 0 and ((k + 1) % reorthonormalize_every == 0):
            U_new = orthonormalize_frames_torch(U_new, eps=eps)

        I = I + M_scalar * dW
        X = X_new
        U = U_new

    fvals = f_fn(X, y, **f_kwargs)

    if fvals.shape != (B, P):
        raise ValueError(
            f"f_fn must return shape {(B, P)}, but got {tuple(fvals.shape)}"
        )

    fvals = fvals.to(dtype=dtype, device=device)

    sum_f = fvals.sum(dim=1)                      # (B,)
    sum_fI = (fvals.unsqueeze(-1) * I).sum(dim=1)  # (B,2)

    return sum_f, sum_fI


# ============================================================
# Chunked batched BEL estimator
# ============================================================

@torch.no_grad()
def bel_gradlog_u_S2_batch_chunked_torch(
    X0, y, t,
    f_fn,
    n_paths=8000,
    n_steps=200,
    chunk_size=1024,
    seed=0,
    device=None,
    dtype=DEFAULT_DTYPE,
    reorthonormalize_every=1,
    empty_cache_between_chunks=False,
    f_kwargs=None,
    grad_only  = True
):
    """
    Generalized chunked BEL estimator.

    Estimates:
      u(t,x0) = E[f_fn(X_t, y)]
      grad log u(t,x0)

    Inputs:
      X0: (B,3)
      y: object consumed by f_fn, typically (3,)
      f_fn: callable:
            f_fn(X, y, **f_kwargs) -> (B,P)
      f_kwargs: optional dict passed into f_fn
    Returns:
      u_hat:   (B,)
      gradlog: (B,3)
    """
    if f_kwargs is None:
        f_kwargs = {}

    if device is None:
        device = get_best_device()

    X0 = torch.as_tensor(X0, dtype=dtype, device=device)
    y = torch.as_tensor(y, dtype=dtype, device=device)

    eps = get_eps(dtype)

    X0 = normalize_torch(X0, eps=eps)
    U0 = make_tangent_frame_S2_batch_torch(X0, eps=eps)

    B = X0.shape[0]
    total_sum_f = torch.zeros(B, dtype=dtype, device=device)
    total_sum_fI = torch.zeros(B, 2, dtype=dtype, device=device)

    n_full = n_paths // chunk_size
    remainder = n_paths % chunk_size

    generator = torch.Generator(device=device)
    generator.manual_seed(seed)

    for _ in range(n_full):
        sum_f, sum_fI = _bel_chunk_kernel_torch(
            X0=X0,
            y=y,
            t=t,
            f_fn=f_fn,
            chunk_size=chunk_size,
            n_steps=n_steps,
            generator=generator,
            reorthonormalize_every=reorthonormalize_every,
            f_kwargs=f_kwargs,
        )
        total_sum_f += sum_f
        total_sum_fI += sum_fI

        if empty_cache_between_chunks and device.type == "cuda":
            torch.cuda.empty_cache()
        elif empty_cache_between_chunks and device.type == "mps":
            torch.mps.empty_cache()

    if remainder > 0:
        sum_f, sum_fI = _bel_chunk_kernel_torch(
            X0=X0,
            y=y,
            t=t,
            f_fn=f_fn,
            chunk_size=remainder,
            n_steps=n_steps,
            generator=generator,
            reorthonormalize_every=reorthonormalize_every,
            f_kwargs=f_kwargs,
        )
        total_sum_f += sum_f
        total_sum_fI += sum_fI

    t = torch.as_tensor(t, dtype=dtype, device=device)
    n_paths_t = torch.as_tensor(float(n_paths), dtype=dtype, device=device)

    u_hat = total_sum_f / n_paths_t

    denom = t * total_sum_f.unsqueeze(-1) + 1e-30
    gradlog_intr = total_sum_fI / denom

    gradlog = torch.einsum("bij,bj->bi", U0, gradlog_intr)
    gradlog = gradlog - torch.sum(gradlog * X0, dim=-1, keepdim=True) * X0

    return (u_hat, gradlog) if grad_only == False else gradlog


# ============================================================
# Single-point wrapper
# ============================================================

@torch.no_grad()
def bel_gradlog_u_S2_chunked_torch(
    x0, y, t,
    f_fn,
    n_paths=20000,
    n_steps=200,
    chunk_size=1024,
    seed=0,
    device=None,
    dtype=DEFAULT_DTYPE,
    reorthonormalize_every=1,
    f_kwargs=None,
    grad_only = True
):
    if device is None:
        device = get_best_device()

    x0 = torch.as_tensor(x0, dtype=dtype, device=device).reshape(1, 3)

    u, g = bel_gradlog_u_S2_batch_chunked_torch(
        X0=x0,
        y=y,
        t=t,
        f_fn=f_fn,
        n_paths=n_paths,
        n_steps=n_steps,
        chunk_size=chunk_size,
        seed=seed,
        device=device,
        dtype=dtype,
        reorthonormalize_every=reorthonormalize_every,
        f_kwargs=f_kwargs,
        grad_only= False
    )
    return (u[0], g[0]) if grad_only == False else g

In [3]:
device = get_best_device()
dtype = recommended_dtype_for_device(device)

x0 = torch.tensor([0., 0., 1.], dtype=dtype, device=device)
y = torch.tensor([1., 0., 0.], dtype=dtype, device=device)

gradlog_hat = bel_gradlog_u_S2_chunked_torch(
    x0=x0,
    y=y,
    t=0.2,
    f_fn=default_gaussian_kernel_f,
    n_paths=20000,
    n_steps=300,
    chunk_size=2048,
    seed=0,
    device=device,
    dtype=dtype,
    f_kwargs={"alpha": 5.0, "euclidean": True},
    grad_only = True
)

#print("u_hat:", float(u_hat.item()))
print(gradlog_hat)
#print("gradlog_hat:", gradlog_hat.cpu().numpy())

tensor([[ 4.8210, -0.0165,  0.0000]], device='mps:0')


In [4]:
# ============================================================
#  integrator: return full path on a time grid
# ============================================================

@torch.no_grad()
def GRW_SDE_path_integrator(
    b,
    x: torch.Tensor,
    T: float,
    n_steps: int = 5,
    generator=None,
    return_path: bool = True,
):
    """
    Geodesic random walk SDE integrator on S^2.

    Args:
        b:
            drift function with signature b(t, x), returning shape like x
        x:
            (d,) or (N, d)
        T:
            terminal time
        n_steps:
            number of integration steps
        generator:
            torch random generator
        return_path:
            if True, return all intermediate states on the grid
            if False, return only x_T

    Returns:
        if return_path:
            path: (n_steps, N, d)
                  where path[k-1] is the state at time t_k = k*T/n_steps
        else:
            xT: (N, d)
    """
    x = normalize_torch(x)

    if x.ndim == 1:
        x = x[None, :]

    N, d = x.shape
    dt = T / n_steps

    if return_path:
        path = []

    for k in range(n_steps):
        tk_mid = dt * (k + 0.5)

        Z = torch.randn((N, d), device=x.device, dtype=x.dtype, generator=generator)
        V = (dt ** 0.5) * Z
        V = tangent_project_torch(x, V) + dt * b(tk_mid, x)
        x = sphere_exp_map_torch(x, V)

        if return_path:
            path.append(x.clone())

    if return_path:
        return torch.stack(path, dim=0)   # (K, N, d)

    return x



GRW_SDE_path_integrator(lambda t, x: bel_gradlog_u_S2_chunked_torch(x0 = x, y = y, t=t, f_fn=default_gaussian_kernel_f), x = x0, T = 1)

tensor([[[ 0.6982, -0.6673,  0.2592]],

        [[ 0.9177, -0.1891,  0.3494]],

        [[ 0.7402, -0.3322,  0.5846]],

        [[ 0.7597,  0.2398,  0.6045]],

        [[ 0.7154,  0.4556,  0.5297]]], device='mps:0')

In [5]:
# ============================================================
# Vector field generator
# ============================================================


def ambient_generators_s2(x: torch.Tensor) -> torch.Tensor:
    """
    X_i(x) = (I - x x^T)e_i
    returns shape (..., 3, 3)
    """
    x = normalize_torch(x)
    eye = torch.eye(3, device=x.device, dtype=x.dtype)
    batch_shape = x.shape[:-1]
    eye = eye.expand(*batch_shape, 3, 3)
    xxT = x.unsqueeze(-1) * x.unsqueeze(-2)
    return eye - xxT


# ============================================================
# model: concat embedding + sinusoidal activation + ambient generator
# ============================================================

class ConcatEmbedding(nn.Module):
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        if t.ndim == x.ndim - 1:
            t = t.unsqueeze(-1)
        return torch.cat([x, t], dim=-1)


class SineLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, bias: bool = True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sin(self.linear(x))


class SineMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, n_hidden_layers: int = 3):
        super().__init__()
        layers = [SineLayer(in_dim, hidden_dim)]
        for _ in range(n_hidden_layers - 1):
            layers.append(SineLayer(hidden_dim, hidden_dim))
        self.hidden = nn.Sequential(*layers)
        self.final = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.final(self.hidden(x))


class AmbientGeneratorScoreNet(nn.Module):
    """
    s_theta(x,t) = sum_i f_i(x,t) X_i(x),
    with X_i(x) = (I - x x^T)e_i.
    """
    def __init__(self, hidden_dim: int = 128, n_hidden_layers: int = 3):
        super().__init__()
        self.embedding = ConcatEmbedding()
        self.net = SineMLP(
            in_dim=4,          # 3 coordinates + 1 scalar time
            hidden_dim=hidden_dim,
            out_dim=3,         # 3 coefficient functions
            n_hidden_layers=n_hidden_layers,
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = normalize_torch(x)
        feat = self.embedding(x, t)
        fi = self.net(feat)                     # (N, 3)
        Xi = ambient_generators_s2(x)           # (N, 3, 3)
        score = torch.einsum("ni,ndi->nd", fi, Xi)
        return tangent_project_torch(x, score)



# ============================================================
# Hutchinson spherical divergence
# ============================================================

def _make_probe(x: torch.Tensor, kind: str = "rademacher", tangent: bool = False) -> torch.Tensor:
    if kind == "rademacher":
        z = (torch.randint(0, 2, x.shape, device=x.device, dtype=torch.int64) * 2 - 1).to(x.dtype)
    elif kind == "gaussian":
        z = torch.randn_like(x)
    else:
        raise ValueError(f"Unknown probe kind: {kind}")

    if tangent:
        z = tangent_project_torch(x, z)
    return z


def spherical_divergence_hutchinson_s2(
    field_fn,
    x: torch.Tensor,
    t: torch.Tensor,
    n_probe: int = 1,
    probe_type: str = "rademacher",
    tangent_probe: bool = False,
    create_graph: bool = True,
) -> torch.Tensor:
    """
    div_{S^2} V = tr(P_x D V) - 2 <x, V>
    estimated with Hutchinson.
    """
    x = normalize_torch(x)
    if t.ndim == 1:
        t = t.unsqueeze(-1)

    x_req = x.detach().clone().requires_grad_(True)
    V = field_fn(x_req, t)  # (N, d)

    trace_est = 0.0
    for _ in range(n_probe):
        z = _make_probe(x_req, kind=probe_type, tangent=tangent_probe)
        Pz = tangent_project_torch(x_req, z)

        inner = (V * Pz).sum(dim=-1)  # (N,)
        grad_inner = torch.autograd.grad(
            outputs=inner.sum(),
            inputs=x_req,
            create_graph=create_graph,
            retain_graph=True,
            only_inputs=True,
        )[0]

        est = (grad_inner * z).sum(dim=-1)
        trace_est = trace_est + est

    trace_est = trace_est / float(n_probe)
    correction = 2.0 * (x_req * V).sum(dim=-1)
    return trace_est - correction


# ============================================================
# one training step using sampled time-indices
# ============================================================

def ism_training_step_pathwise(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    x0: torch.Tensor,
    T: float,
    eps: float,
    n_steps: int,
    n_probe: int = 1,
    probe_type: str = "rademacher",
    tangent_probe: bool = False,
    generator=None,
):
    """
    Training step:
      1) sample a time index for each sample
      2) simulate the full path on a fixed grid once
      3) gather x_t for each sample from the path
      4) compute implicit score matching loss
      5) optimizer step

    Args:
        x0: (N, d) or (d,)
    Returns:
        loss_value, x_t.detach(), t.detach()
    """
    model.train()

    if x0.ndim == 1:
        x0 = x0.unsqueeze(0)

    x0 = normalize_torch(x0)
    device = x0.device
    N = x0.shape[0]

    # ------------------------------------------------------------
    # sample time indices uniformly from {1, ..., n_steps}
    # this approximates t ~ Uniform(eps, T) on a grid
    # ------------------------------------------------------------
    # times are t_k = eps + (T-eps) * k / n_steps, k=1,...,n_steps
    k_idx = torch.randint(
        low=1,
        high=n_steps + 1,
        size=(N,),
        device=device,
        generator=generator,
    )
    t = eps + (T - eps) * (k_idx.float() / n_steps)   # (N,)
    t = t.unsqueeze(-1)                               # (N, 1)

    # ------------------------------------------------------------
    # simulate full path once on the batch
    # ------------------------------------------------------------
    with torch.no_grad():
        zero_drift = lambda tau, x: torch.zeros_like(x)
        path = GRW_SDE_path_integrator(
            zero_drift,
            x0,
            T=T,
            n_steps=n_steps,
            generator=generator,
            return_path=True,
        )  # (K, N, d)

        # gather x_t for each sample according to sampled time index
        # path index is k_idx - 1
        x_t = path[k_idx - 1, torch.arange(N, device=device)]  # (N, d)

    # ------------------------------------------------------------
    # implicit score matching loss
    # ------------------------------------------------------------
    score = model(x_t, t)
    sq_norm = (score * score).sum(dim=-1)

    div_score = spherical_divergence_hutchinson_s2(
        model,
        x_t,
        t,
        n_probe=n_probe,
        probe_type=probe_type,
        tangent_probe=tangent_probe,
        create_graph=True,
    )

    loss = (0.5 * sq_norm + div_score).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return loss.item(), x_t.detach(), t.detach()


# ============================================================
# full training loop
# ============================================================

def train_ism_pathwise(
    model: nn.Module,
    data_loader,
    T: float = 1.0,
    eps: float = 1e-3,
    lr: float = 1e-3,
    n_epochs: int = 20,
    n_steps: int = 16,
    n_probe: int = 1,
    probe_type: str = "rademacher",
    tangent_probe: bool = False,
    device: str = "cpu",
    generator=None,
    log_every: int = 50,
):
    """
    Full training loop for ISM on the sphere.

    Assumes each batch from data_loader is either:
      - x
      - (x, ...)
    where x are samples from p_data.

    Returns:
      trained model
    """
    device = torch.device(device)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    global_step = 0
    loss_history = []

    for epoch in range(n_epochs):
        for batch in data_loader:
            if isinstance(batch, (list, tuple)):
                x0 = batch[0]
            else:
                x0 = batch

            x0 = x0.to(device)

            loss_value, x_t, t = ism_training_step_pathwise(
                model=model,
                optimizer=optimizer,
                x0=x0,
                T=T,
                eps=eps,
                n_steps=n_steps,
                n_probe=n_probe,
                probe_type=probe_type,
                tangent_probe=tangent_probe,
                generator=generator,
            )

            loss_history.append(loss_value)

            if global_step % log_every == 0:
                print(
                    f"epoch={epoch:03d} "
                    f"step={global_step:06d} "
                    f"loss={loss_value:.6f} "
                    f"t_mean={t.mean().item():.4f}"
                )

            global_step += 1

    model.loss_history = loss_history
    return model


# ============================================================
# example usage
# ============================================================
#
# model = AmbientGeneratorScoreNet(hidden_dim=128, n_hidden_layers=3)
#
# trained_model = train_ism_pathwise(
#     model=model,
#     data_loader=train_loader,         # batches of x0 ~ p_data
#     T=1.0,
#     eps=1e-3,
#     lr=1e-3,
#     n_epochs=20,
#     n_steps=16,                       # size of time grid + SDE steps
#     n_probe=4,
#     probe_type="rademacher",
#     tangent_probe=False,
#     device="cuda" if torch.cuda.is_available() else "cpu",
#     log_every=10,
# )

In [6]:
import os
from torch.utils.data import Dataset, DataLoader, random_split

# ============================================================
# sphere helpers
# ============================================================

def normalize_torch(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def latlon_deg_to_extrinsic_torch(latlon_deg: torch.Tensor) -> torch.Tensor:
    """
    Matches the uploaded earth.py logic:

        intrinsic = pi * (data / 180) + [pi/2, pi]
        extrinsic = spherical_to_extrinsic(intrinsic)

    Input:
        latlon_deg: (N, 2), columns [latitude_deg, longitude_deg]

    Returns:
        x: (N, 3) points on S^2 in extrinsic coordinates
    """
    intrinsic = torch.pi * (latlon_deg / 180.0) + torch.tensor(
        [torch.pi / 2, torch.pi], dtype=latlon_deg.dtype, device=latlon_deg.device
    )

    theta = intrinsic[:, 0]
    phi = intrinsic[:, 1]

    x = torch.sin(theta) * torch.cos(phi)
    y = torch.sin(theta) * torch.sin(phi)
    z = torch.cos(theta)

    out = torch.stack([x, y, z], dim=-1)
    return normalize_torch(out)


# ============================================================
# earth dataset
# mirrors the uploaded earth.py structure
# ============================================================

class EarthSphericalDataset(Dataset):
    """
    Torch version of the uploaded earth.py spherical dataset loader.

    The uploaded code loads lat/lon CSV data and converts it to S^2 points. :contentReference[oaicite:0]{index=0}
    """
    FILE_MAP = {
        "earthquake": ("quakes_all.csv", 4),
        "fire": ("fire.csv", 1),
        "flood": ("flood.csv", 2),
        "volcano": ("volerup.csv", 2),
    }

    def __init__(
        self,
        data_dir: str = "data",
        name: str = "earthquake",
        dtype: torch.dtype = torch.float32,
        device: str = "cpu",
    ):
        super().__init__()

        if name not in self.FILE_MAP:
            raise ValueError(
                f"Unknown earth dataset '{name}'. "
                f"Choose from {list(self.FILE_MAP.keys())}."
            )

        filename, skip_header = self.FILE_MAP[name]
        path = os.path.join(data_dir, filename)

        if not os.path.exists(path):
            raise FileNotFoundError(f"Could not find dataset file: {path}")

        raw = np.genfromtxt(path, delimiter=",", skip_header=skip_header)
        raw = np.asarray(raw, dtype=np.float32)

        if raw.ndim != 2 or raw.shape[1] < 2:
            raise ValueError(
                f"Expected a 2D CSV with at least 2 columns for lat/lon, got shape {raw.shape}"
            )

        # same convention as uploaded earth.py: use first two columns as lat/lon
        latlon_deg = torch.tensor(raw[:, :2], dtype=dtype, device=device)
        extrinsic = latlon_deg_to_extrinsic_torch(latlon_deg)

        self.name = name
        self.data_dir = data_dir
        self.latlon_deg = latlon_deg
        self.data = extrinsic

    def __len__(self) -> int:
        return self.data.shape[0]

    def __getitem__(self, idx: int):
        # match your training code convention: return (x, context)
        x = self.data[idx]
        context = None
        return x, context


# ============================================================
# collate function
# keeps the interface (x, context)
# ============================================================

def earth_collate_fn(batch):
    xs = torch.stack([item[0] for item in batch], dim=0)
    contexts = [item[1] for item in batch]
    context = None if all(c is None for c in contexts) else contexts
    return xs, context


# ============================================================
# convenience builder
# ============================================================

def make_earth_dataloaders(
    data_dir: str = "data",
    name: str = "earthquake",
    batch_size: int = 512,
    eval_batch_size: int | None = None,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    seed: int = 0,
    shuffle_train: bool = True,
    num_workers: int = 0,
    pin_memory: bool = False,
    dtype: torch.dtype = torch.float32,
    device: str = "cpu",
):
    """
    Returns:
        train_loader, val_loader, test_loader, full_dataset
    """
    if eval_batch_size is None:
        eval_batch_size = batch_size

    dataset = EarthSphericalDataset(
        data_dir=data_dir,
        name=name,
        dtype=dtype,
        device=device,
    )

    n = len(dataset)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    n_test = n - n_train - n_val

    g = torch.Generator()
    g.manual_seed(seed)

    train_ds, val_ds, test_ds = random_split(
        dataset,
        lengths=[n_train, n_val, n_test],
        generator=g,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=earth_collate_fn,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=earth_collate_fn,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=eval_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=earth_collate_fn,
    )

    return train_loader, val_loader, test_loader, dataset


# ============================================================
# example usage
# ============================================================
if __name__ == "__main__":
    train_loader, val_loader, test_loader, dataset = make_earth_dataloaders(
        data_dir="data",
        name="earthquake",   # "fire", "flood", "volcano"
        batch_size=256,
        eval_batch_size=512,
        seed=0,
        device="cpu",
    )

    x, context = next(iter(train_loader))
    print("dataset:", dataset.name)
    print("batch shape:", x.shape)              # (B, 3)
    print("context:", context)                  # None
    print("norm check:", x.norm(dim=-1).mean()) # ~1

dataset: earthquake
batch shape: torch.Size([256, 3])
context: None
norm check: tensor(1.)
